In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
!cp -r /kaggle/input/rasoul-multiclass-obb/dataset /kaggle/working/dataset

In [3]:
def convert_obb_annotations_to_binary(root_label_dir, new_class_id=0):
    """
    Converts all YOLOv8 OBB annotations in train/val/test subfolders to binary class ID (default = 0).
    Modifies files in-place.

    Args:
        root_label_dir (str): Path to the root directory containing train/val/test label folders.
        new_class_id (int): The ID to assign to all annotations (default = 0).
    """
    for split in ["train", "val", "test"]:
        split_dir = os.path.join(root_label_dir, split)
        if not os.path.exists(split_dir):
            print(f"⚠️ Missing folder: {split_dir}")
            continue

        for filename in os.listdir(split_dir):
            if filename.endswith(".txt"):
                file_path = os.path.join(split_dir, filename)
                with open(file_path, 'r') as file:
                    lines = file.readlines()

                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) == 9:  # YOLOv8 OBB format: class_id x1 y1 ... x4 y4
                        parts[0] = str(new_class_id)
                        new_lines.append(" ".join(parts))

                with open(file_path, 'w') as file:
                    file.write("\n".join(new_lines))

        print(f"✅ Converted annotations in: {split_dir}")

# === Example usage ===
convert_obb_annotations_to_binary("/kaggle/working/dataset/labels")

✅ Converted annotations in: /kaggle/working/dataset/labels/train
✅ Converted annotations in: /kaggle/working/dataset/labels/val
✅ Converted annotations in: /kaggle/working/dataset/labels/test


In [4]:
def visualize_yolo_obb(image_path, label_path):
    """
    Visualizes YOLOv8 OBB annotations on a given image.
    
    Args:
        image_path (str): Path to the image.
        label_path (str): Path to the corresponding YOLO OBB label file.
    """
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Failed to load image: {image_path}")
        return
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]

    # Read label file
    if not os.path.exists(label_path):
        print(f"❌ Label file not found: {label_path}")
        return

    with open(label_path, 'r') as f:
        lines = f.readlines()

    # Draw OBBs
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 9:
            continue
        class_id = int(parts[0])
        coords = np.array([float(p) for p in parts[1:]], dtype=np.float32).reshape(-1, 2)
        coords[:, 0] *= width
        coords[:, 1] *= height
        coords = coords.astype(np.int32)

        cv2.polylines(image, [coords], isClosed=True, color=(255, 0, 0), thickness=2)
        cx = int(coords[:, 0].mean())
        cy = int(coords[:, 1].mean())
        cv2.putText(image, str(class_id), (cx, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    # Show image
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"{os.path.basename(image_path)}")
    plt.savefig(f"/kaggle/working/samples/{os.path.basename(image_path)}")
    plt.show()

In [5]:
os.makedirs("/kaggle/working/samples", exist_ok=True)
visualize_yolo_obb(
    image_path="/kaggle/working/dataset/images/test/KORD MEHDI-638573282651304435_8.png",
    label_path="/kaggle/working/dataset/labels/test/KORD MEHDI-638573282651304435_8.txt"
)

❌ Failed to load image: /kaggle/working/dataset/images/test/KORD MEHDI-638573282651304435_8.png


[ WARN:0@103.540] global loadsave.cpp:268 findDecoder imread_('/kaggle/working/dataset/images/test/KORD MEHDI-638573282651304435_8.png'): can't open/read file: check file path/integrity


In [6]:
visualize_yolo_obb(
    image_path="/kaggle/working/dataset/images/test/RANJBAR FATEME-638573376220046811_4.png",
    label_path="/kaggle/working/dataset/labels/test/RANJBAR FATEME-638573376220046811_4.txt"
)

❌ Failed to load image: /kaggle/working/dataset/images/test/RANJBAR FATEME-638573376220046811_4.png


[ WARN:0@103.554] global loadsave.cpp:268 findDecoder imread_('/kaggle/working/dataset/images/test/RANJBAR FATEME-638573376220046811_4.png'): can't open/read file: check file path/integrity


In [7]:
visualize_yolo_obb(
    image_path="/kaggle/working/dataset/images/test/KARDEH FATEMEH-638573373382572098_23.png",
    label_path="/kaggle/working/dataset/labels/test/KARDEH FATEMEH-638573373382572098_23.txt"
)

❌ Failed to load image: /kaggle/working/dataset/images/test/KARDEH FATEMEH-638573373382572098_23.png


[ WARN:0@103.567] global loadsave.cpp:268 findDecoder imread_('/kaggle/working/dataset/images/test/KARDEH FATEMEH-638573373382572098_23.png'): can't open/read file: check file path/integrity
